# Projeto - Vendas - SQL/Python


In [ ]:
# Vamos instalar o pacote watermark para gerar uma marca d'água com as informações do ambiente em execução

!pip install -q -U watermark 

In [ ]:
# Instalação do pandas para manipulação de dados 

!pip install pandas 

In [ ]:
# Pacote que permite usar sql dentro do python para realizar consultas 

!pip install -q -U ipython-sql 

In [4]:
# Imports 

import os                   # Biblioteca para acessar o sistema operacional 
import pandas   as pd       # Biblioteca para manipular dados em python 
import sqlite3              # Banco de dados super leve 

In [5]:
# Carregamos a extensão watermark 

%reload_ext watermark 
%watermark -a "Victor Rufato"  -v -m -d -p ipython-sql,pandas

Author: Victor Rufato

Date: 2026-07-09

Python implementation: CPython
Python version       : 3.12.0
IPython version      : 9.15.0

ipython-sql: 0.5.0
pandas     : 3.0.3

Compiler    : MSC v.1935 64 bit (AMD64)
OS          : Windows
Release     : 11
Machine     : AMD64
Processor   : Intel64 Family 6 Model 140 Stepping 1, GenuineIntel
CPU cores   : 8
Architecture: 64bit



#### Extração dos dados 

In [6]:
# Extrai dados do arquivo e salva em um dataframe do pandas 

dados = pd.read_csv('base_vendas.csv')

In [7]:
# Verifica se é um dataframe do pandas 

type(dados)

pandas.DataFrame

In [8]:
# Verificamos a quantidade de linhas e colunas da tabela 

dados.shape

(1520, 24)

In [9]:
# Retorna as 5 primeiras linhas da tabela 

dados.head()

,id_pedido,data_venda,id_cliente,cliente,segmento,regiao,uf,id_produto,produto,categoria,...,preco_unitario,desconto_pct,receita_bruta,valor_desconto,receita_liquida,custo_total,lucro,status_pedido,prazo_entrega_dias,avaliacao_cliente
0,V00001,2025-10-09,C002,Beta Varejo,B2B,Sudeste,MG,P001,Notebook Pro 14,Informática,...,4658.46,0.05,9316.92,465.85,8851.07,6269.32,2581.75,Concluída,3,2.0
1,V00002,2025-03-31,C002,Beta Varejo,B2B,Sudeste,MG,P009,Smartphone X,Telefonia,...,2890.29,0.05,17341.74,867.09,16474.65,10973.64,5501.01,Pendente,7,3.0
2,V00003,2025-08-27,C002,Beta Varejo,B2B,Sudeste,MG,P004,Monitor 24,Informática,...,836.91,0.00,836.91,0.00,836.91,618.82,218.09,Concluída,3,5.0
3,V00004,2025-08-14,C009,Shopping Mix,B2C,Nordeste,PE,P007,Mesa Office,Móveis,...,752.13,0.00,752.13,0.00,752.13,430.90,321.23,Cancelada,3,4.0
4,V00005,2026-03-09,C001,Alfa Distribuidora,B2B,Sudeste,SP,P003,Teclado Mecânico,Acessórios,...,291.27,0.05,873.81,43.69,830.12,413.55,416.57,Concluída,10,NaN


#### Criação e conexão do banco de dados 

In [10]:
# Vamos definir onde será o caminho onde vamos salvar nosso banco de dados

arquivo_path = 'database.db'

In [ ]:
# Verifica se o arquivo existe e deleta se existir para criar um novo arquivo posteriormente 

# Try e except evitam que o programa quebre quando acontece erro, são para tratar erros

if os.path.exists(arquivo_path):
    try:
        os.remove(arquivo_path)
        print("Arquivo removido com sucesso")
    except Exception as e:
        print(f"Erro ao deletar o {arquivo_path}. Detalhes: {e}")
else:
    print(f"{arquivo_path} não encontrado")

In [12]:
# Vamos agora criar uma conexão com um banco de dados sqlite3

cnn = sqlite3.connect('database.db')

In [13]:
# Direciono meu dataframe para o banco de dados renomeando minha tabela para vendas 

dados.to_sql('vendas',cnn)

1520

In [14]:
# Vamos agora carregar a extensão para usar os comandos sql 

%load_ext sql
%sql sqlite:///database.db

In [15]:
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

#### Consultas SQL 



#### Solicitações do gestor 

- Apenas pedidos Concluídos
- Sem duplicidades de id_pedido
- Registros inválidos removidos:
        quantidade <= 0;
        preco_unitario <= 0;
        receita_liquida <= 0;

Colunas finais:

ano_mes
regiao
uf
categoria
produto
canal_venda
vendedor
qtd_pedidos
qtd_itens_vendidos
receita_liquida_total
custo_total
lucro_total
margem_lucro_pct
ticket_medio


Vamos verificar quais id_pedido tem registros duplicados e depois removê-los 

In [ ]:
%%sql

SELECT
    id_pedido
    ,COUNT(*)                                                   AS qtd_registros
FROM vendas
GROUP BY id_pedido
ORDER BY qtd_registros DESC 

In [ ]:
%%sql


WITH duplicatas AS (
    SELECT
        strftime('%Y/%m', data_venda) 												AS ano_mes,
        regiao,
        uf,
        categoria,
        produto,
        canal_venda,
        vendedor,
        id_pedido,
        quantidade,
        receita_liquida,
        custo_total,
        lucro,
        ROW_NUMBER() OVER (
            PARTITION BY id_pedido
            ORDER BY data_venda
        ) AS rn
    FROM vendas
    WHERE status_pedido IN ('Concluída', 'Concluido')
      AND quantidade > 0
      AND preco_unitario > 0
      AND receita_liquida > 0
)
SELECT
    ano_mes,
    regiao,
    uf,
    categoria,
    produto,
    canal_venda,
    vendedor,
    COUNT(id_pedido) 																AS qtd_pedidos,
    SUM(quantidade) 																AS qtd_itens_vendidos,
    SUM(receita_liquida) 															AS receita_liquida_total,
    SUM(custo_total) 																AS custo_total,
    SUM(lucro) 																		AS lucro_total,
    ROUND((SUM(lucro) / SUM(receita_liquida)) * 100, 2) 							AS margem_lucro_pct,
    ROUND(SUM(receita_liquida) / COUNT(id_pedido), 2) 								AS ticket_medio
FROM duplicatas
WHERE rn = 1
GROUP BY
    ano_mes,
    regiao,
    uf,
    categoria,
    produto,
    canal_venda,
    vendedor;

In [ ]:
%%sql

DROP TABLE IF EXISTS vendas_processado;

CREATE TABLE vendas_processado (
  ano_mes CHAR(7),                  
  regiao VARCHAR(30),
  uf CHAR(2),
  categoria VARCHAR(50),
  produto VARCHAR(100),
  canal_venda VARCHAR(50),
  vendedor VARCHAR(100),
  qtd_pedidos INT,
  qtd_itens_vendidos INT,
  receita_liquida_total DECIMAL(15,2),
  custo_total DECIMAL(15,2),
  lucro_total DECIMAL(15,2),
  margem_lucro_pct DECIMAL(5,2),
  ticket_medio DECIMAL(15,2)
);

In [ ]:
%%sql

INSERT INTO vendas_processado (
    ano_mes, regiao, uf, categoria, produto, canal_venda, vendedor, qtd_pedidos, qtd_itens_vendidos, receita_liquida_total, custo_total, lucro_total, margem_lucro_pct, ticket_medio)
WITH duplicatas AS (
    SELECT
        strftime('%Y/%m', data_venda)                                       AS ano_mes,
        regiao,
        uf,
        categoria,
        produto,
        canal_venda,
        vendedor,
        id_pedido,
        quantidade,
        receita_liquida,
        custo_total,
        lucro,
        ROW_NUMBER() OVER (
            PARTITION BY id_pedido
            ORDER BY data_venda
        ) AS rn
    FROM vendas
    WHERE status_pedido IN ('Concluída', 'Concluido')
      AND quantidade > 0
      AND preco_unitario > 0
      AND receita_liquida > 0
)
SELECT
    ano_mes,
    regiao,
    uf,
    categoria,
    produto,
    canal_venda,
    vendedor,
    COUNT(id_pedido)                                                        AS qtd_pedidos,
    SUM(quantidade)                                                         AS qtd_itens_vendidos,
    SUM(receita_liquida)                                                    AS receita_liquida_total,
    SUM(custo_total)                                                        AS custo_total,
    SUM(lucro)                                                              AS lucro_total,
    ROUND((SUM(lucro) / SUM(receita_liquida)) * 100, 2)                     AS margem_lucro_pct,
    ROUND(SUM(receita_liquida) / COUNT(id_pedido), 2)                       AS ticket_medio
FROM duplicatas
WHERE rn = 1
GROUP BY
    ano_mes,
    regiao,
    uf,
    categoria,
    produto,
    canal_venda,
    vendedor;

In [ ]:
%%sql

SELECT * FROM vendas_processado


In [21]:
# Query

dados_query = cnn.execute("SELECT * FROM vendas_processado")

In [22]:
dados_query

# Cursor é um objeto que permite percorrer por todos os dados da tabela

In [23]:
dados_query.description

(('ano_mes', None, None, None, None, None, None),
 ('regiao', None, None, None, None, None, None),
 ('uf', None, None, None, None, None, None),
 ('categoria', None, None, None, None, None, None),
 ('produto', None, None, None, None, None, None),
 ('canal_venda', None, None, None, None, None, None),
 ('vendedor', None, None, None, None, None, None),
 ('qtd_pedidos', None, None, None, None, None, None),
 ('qtd_itens_vendidos', None, None, None, None, None, None),
 ('receita_liquida_total', None, None, None, None, None, None),
 ('custo_total', None, None, None, None, None, None),
 ('lucro_total', None, None, None, None, None, None),
 ('margem_lucro_pct', None, None, None, None, None, None),
 ('ticket_medio', None, None, None, None, None, None))

In [24]:
# List Comprehension para retornar os metadados da tabela

cols = [coluna[0] for coluna in dados_query.description]

In [25]:
cols


['ano_mes',
 'regiao',
 'uf',
 'categoria',
 'produto',
 'canal_venda',
 'vendedor',
 'qtd_pedidos',
 'qtd_itens_vendidos',
 'receita_liquida_total',
 'custo_total',
 'lucro_total',
 'margem_lucro_pct',
 'ticket_medio']

In [26]:
resultado = pd.DataFrame.from_records(data=dados_query.fetchall(),columns=cols)

In [27]:
resultado.shape

(1137, 14)

In [28]:
resultado.head()

,ano_mes,regiao,uf,categoria,produto,canal_venda,vendedor,qtd_pedidos,qtd_itens_vendidos,receita_liquida_total,custo_total,lucro_total,margem_lucro_pct,ticket_medio
0,2025/01,Centro-Oeste,GO,Móveis,Mesa Office,Marketplace,Diego Alves,1,1,689.55,453.52,236.03,34.23,689.55
1,2025/01,Centro-Oeste,GO,Telefonia,Smartphone X,Loja Física,Diego Alves,1,2,5259.33,4129.46,1129.87,21.48,5259.33
2,2025/01,Centro-Oeste,GO,Telefonia,Smartphone X,Marketplace,Diego Alves,1,2,4916.08,4213.02,703.06,14.30,4916.08
3,2025/01,Nordeste,BA,Acessórios,Headset Gamer,Loja Física,Ana Souza,1,10,2352.99,972.20,1380.79,58.68,2352.99
4,2025/01,Nordeste,BA,Acessórios,Mouse Wireless,E-commerce,Diego Alves,1,1,98.23,36.06,62.17,63.29,98.23


In [29]:
## Salvando os dados em um arquivo csv

resultado.to_csv('resultado.csv',index=False)